In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchWindowException
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from datetime import datetime
import os
from dotenv import load_dotenv
from time import sleep

load_dotenv()

# ======= CONFIGURAÇÕES INICIAIS =======
def configuracoes_chrome():
    chrome_options = Options()
    chrome_options.add_argument("--log-level=3")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
    chrome_options.add_argument("--start-maximized")
    return chrome_options


config = {
    "usuario": os.getenv("USUARIO"),
    "senha": os.getenv("SENHA"),
    "natureza": ["11"],
    "filiais_excluidas": ["SAO PAULO", "RECIFE"],
    "url_brudam": "https://vdclog.brudam.com.br/financeiro/contas_pagar.php?"
}

# ======= INICIAR NAVEGADOR =======
chrome_options = configuracoes_chrome()
service = Service(log_path="NUL")
browser = webdriver.Chrome(service=service, options=chrome_options)
browser.get(config["url_brudam"])

wdw = WebDriverWait(browser, 10)

# ======= LOGIN =======
wdw.until(EC.element_to_be_clickable((By.ID, "user")))
browser.find_element(By.ID, "user").send_keys(config["usuario"])
browser.find_element(By.ID, "password").send_keys(config["senha"])
browser.find_element(By.ID, "acessar").click()

# ======= ESPERAR CARREGAR =======
wdw.until(EC.presence_of_element_located((By.ID, "form1")))
sleep(3)
# ======= FILTRO DATA =======
data_1 = browser.find_element(By.ID, "data_1")
data_2 = browser.find_element(By.ID, "data_2")
data_1.click()
data_1.send_keys('25/04/2026')
data_2.click()
data_2.send_keys('25/04/2026')

# ======= SITUAÇÃO =======
browser.find_element(By.ID, "selSituacao").click()
marcar = wdw.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input.situacoes[value='1']")))
browser.execute_script("arguments[0].click();", marcar)
browser.find_element(By.CSS_SELECTOR, ".ui-dialog-buttonset").click()

# ======= CENTRO DE CUSTO =======
browser.find_element(By.ID, "selecionaCentros").click()
wdw.until(EC.presence_of_element_located((By.ID, "formularioNovoCentroCusto")))

for item in config["natureza"]:
    browser.find_element(By.CSS_SELECTOR, f"input.check_centro[value='{item}']").click()

browser.find_element(By.XPATH, "//button[text()='Incluir']").click()

# ======= FILIAL =======
browser.find_element(By.ID, "filtro_unidades").click()
wdw.until(EC.presence_of_element_located((By.ID, "formularioUnidades")))

for item in config["filiais_excluidas"]:
    unidades = browser.find_elements(By.CSS_SELECTOR, f"input[textcheck='{item}']")
    for unidade in unidades:
        unidade.click()

browser.find_element(By.XPATH, "//button[text()='Incluir']").click()

# ======= PESQUISAR =======
browser.find_element(By.ID, "PESQUISAR").click()

# ======= AGUARDAR RESULTADOS =======
wdw.until(EC.presence_of_element_located((By.ID, "listaLancamentos")))
linhas = browser.find_elements(By.CLASS_NAME, "LinhaTabela")
aba_principal = browser.current_window_handle

# ======= DATAS PROCURADAS =======
data_procurada = [
    datetime.strptime("17/03/2026", "%d/%m/%Y"),
    datetime.strptime("17/02/2026", "%d/%m/%Y"),
    datetime.strptime("21/02/2026", "%d/%m/%Y"),
    datetime.strptime("23/02/2026", "%d/%m/%Y"),
    datetime.strptime("02/03/2026", "%d/%m/%Y"),
    datetime.strptime("07/03/2026", "%d/%m/%Y"),
    datetime.strptime("09/03/2026", "%d/%m/%Y"),
]

# ======= LOOP PRINCIPAL =======
for linha in linhas:
    id_lancamento = linha.get_attribute("id")

    url = f"https://vdclog.brudam.com.br/financeiro/consulta_lancamento.php?id={id_lancamento}"
    browser.execute_script("window.open(arguments[0]);", url)
    browser.switch_to.window(browser.window_handles[-1])

    fornecedor_nome = browser.find_element(By.ID, "fornecedor_bold").get_attribute("value")

    try:
        browser.find_element(By.ID, "abrirFatura").click()
        browser.switch_to.window(browser.window_handles[-1])

        wdw.until(EC.presence_of_element_located((By.CLASS_NAME, "selecao")))
        selecoes = browser.find_elements(By.CLASS_NAME, "selecao")

        manifestos = []

        for selecao in selecoes:
            lista = selecao.find_elements(By.CLASS_NAME, "LISTA_linha")

            numero = lista[1].text
            data_saida = lista[3].text

            manifestos.append({
                "numero": numero,
                "data": data_saida
            })

        # ======= FILTRO =======
        manifestos_encontrados = []

        for m in manifestos:
            data = m["data"]

            if not data or data.strip() == "":
                continue

            try:
                data_convertida = datetime.strptime(data, "%d/%m/%Y")
            except ValueError:
                continue

            if data_convertida in data_procurada:
                manifestos_encontrados.append({
                    "numero": m["numero"],
                    "data": data_convertida
                })

        # ======= RESULTADO =======
        if manifestos_encontrados:
            resultado = [
                f"{m['numero']} ({m['data'].strftime('%d/%m/%Y')})"
                for m in manifestos_encontrados
            ]
            print(f"{fornecedor_nome} -> Manifestos: {resultado}")
        else:
            print(f"Tudo limpo para o agregado {fornecedor_nome}")

        browser.close()
        browser.switch_to.window(browser.window_handles[-1])

    except TimeoutException:
        print(f"Código {id_lancamento} não localizou a fatura (timeout)")
    except NoSuchWindowException:
        print(f"Código {id_lancamento} perdeu a janela")
    except Exception as e:
        print(f"Erro no código {id_lancamento}: {e}")

    browser.close()
    browser.switch_to.window(aba_principal)

# ======= FINALIZAR =======
browser.quit()

RICARDO DE JESUS -> Manifestos: ['131030 (23/02/2026)']
DILUCIO DE SOUSA CASTANHEIRA FILHO -> Manifestos: ['130905 (21/02/2026)']
VITOR FIDELIX DOS SANTOS -> Manifestos: ['133993 (07/03/2026)', '134157 (09/03/2026)']
FERNANDO DOS SANTOS FAGUNDES -> Manifestos: ['132528 (02/03/2026)', '132621 (02/03/2026)', '133986 (07/03/2026)', '134115 (09/03/2026)']
AECIO RANIERE BRITO DA SILVA -> Manifestos: ['132588 (02/03/2026)', '133979 (07/03/2026)', '134128 (09/03/2026)']
ZCM IMPORTADORA E DISTRIBUIDORA -> Manifestos: ['130810 (21/02/2026)', '130813 (21/02/2026)', '130824 (21/02/2026)', '131146 (23/02/2026)']
RUY LIMA PORTO -> Manifestos: ['132507 (02/03/2026)', '134102 (09/03/2026)']
RUY LIMA PORTO -> Manifestos: ['134028 (07/03/2026)', '134065 (07/03/2026)', '134152 (09/03/2026)', '134221 (09/03/2026)']
Tudo limpo para o agregado BRUNO DA SILVA LEAL
I OLIVEIRA DIAS TRANSPORTES -> Manifestos: ['130939 (21/02/2026)', '131052 (23/02/2026)']
ROSANGELA CORDEIRO DOS SANTOS -> Manifestos: ['130935 (